# GTC Stream Nodes → Neo4j
Pydantic models for the GTC streaming pipeline, upserted into Neo4j.

## 1. Imports

In [ ]:
import os
from typing import ClassVar, List

from neo4j import GraphDatabase
from pydantic import BaseModel, ConfigDict, HttpUrl

## 2. Base model
Every stream node has a `name` and a validated `url`. The `id` is generated by Neo4j (`randomUUID()`) on create. Subclasses set the Neo4j label.

In [ ]:
class StreamNode(BaseModel):
    model_config = ConfigDict(frozen=True)

    label: ClassVar[str]

    name: str
    url: HttpUrl

    def props(self) -> dict:
        return {"name": self.name, "url": str(self.url)}

## 3. Node labels

In [ ]:
class MainInput(StreamNode):
    label: ClassVar[str] = "MainInput"


class BackupLink(StreamNode):
    label: ClassVar[str] = "BackupLink"


class Transcoding(StreamNode):
    label: ClassVar[str] = "Transcoding"


class FinalLink(StreamNode):
    label: ClassVar[str] = "FinalLink"

## 4. Node data

In [ ]:
NODES: List[StreamNode] = [
    MainInput(
        name="GTC Main",
        url="https://jio.ottlive.co.in/gtcnews/gtcnews/index.m3u8",
    ),
    BackupLink(
        name="GTC Backup",
        url="https://ingest.ottlive.co.in/gtcnews/gtcnews/index.m3u8",
    ),
    Transcoding(
        name="GTC Transcoding",
        url="https://ingest.ottlive.co.in/gtcnewsoutput/index.m3u8",
    ),
    FinalLink(
        name="GTC Final",
        url="https://gtc.ottlive.co.in/gtcnews/index.m3u8",
    ),
]

for node in NODES:
    print(node.label, node.props())

MainInput {'name': 'GTC Main', 'url': 'https://jio.ottlive.co.in/gtcnews/gtcnews/index.m3u8'}
BackupLink {'name': 'GTC Backup', 'url': 'https://ingest.ottlive.co.in/gtcnews/gtcnews/index.m3u8'}
Transcoding {'name': 'GTC Transcoding', 'url': 'https://ingest.ottlive.co.in/gtcnewsoutput/index.m3u8'}
FinalLink {'name': 'GTC Final', 'url': 'https://gtc.ottlive.co.in/gtcnews/index.m3u8'}


## 5. Load `.env` credentials

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 6. Connect to Neo4j

In [ ]:
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
print("Connected")

Connected


## 7. Upsert nodes
`MERGE` on `name`, so re-running updates nodes instead of duplicating them. Neo4j sets `id = randomUUID()` only when a node is first created.

In [ ]:
def upsert_nodes(driver, nodes: List[StreamNode]) -> None:
    with driver.session() as session:
        for node in nodes:
            # Labels can't be query parameters; they come from the class, not user input.
            record = session.run(
                f"MERGE (n:{node.label} {{name: $name}}) "
                "ON CREATE SET n.id = randomUUID() "
                "SET n.url = $url "
                "RETURN n.id AS id",
                **node.props(),
            ).single()
            print(node.label, record["id"])


upsert_nodes(driver, NODES)
print(f"Upserted {len(NODES)} nodes")

MainInput aae3ed4d-d4c1-443c-bd64-7f87ed87aed3


BackupLink 1e795438-fed7-476b-82a1-3cb9372d3628
Transcoding cfe52979-bc7b-4f08-8bef-a5340169f023


FinalLink 80a15255-c87c-40aa-84a5-f075f2fe3e9c
Upserted 4 nodes


## 8. Verify

In [ ]:
records = driver.execute_query(
    "MATCH (n) WHERE n.name IN $names RETURN labels(n) AS labels, n.id AS id, n.name AS name, n.url AS url",
    names=[n.name for n in NODES],
).records

for r in records:
    print(r["labels"], r["id"], r["name"], r["url"])

['MainInput'] aae3ed4d-d4c1-443c-bd64-7f87ed87aed3 GTC Main https://jio.ottlive.co.in/gtcnews/gtcnews/index.m3u8
['BackupLink'] 1e795438-fed7-476b-82a1-3cb9372d3628 GTC Backup https://ingest.ottlive.co.in/gtcnews/gtcnews/index.m3u8
['Transcoding'] cfe52979-bc7b-4f08-8bef-a5340169f023 GTC Transcoding https://ingest.ottlive.co.in/gtcnewsoutput/index.m3u8
['FinalLink'] 80a15255-c87c-40aa-84a5-f075f2fe3e9c GTC Final https://gtc.ottlive.co.in/gtcnews/index.m3u8


## 9. Close connection

In [ ]:
driver.close()